In [ ]:
import os
import re
import sys
import io
import logging
from pathlib import Path

import numpy as np
import matplotlib
import socket
import tqdm as tqdm
HOSTNAME = socket.gethostname().lower()
print(f"Running on: {HOSTNAME}")

# Set backend before pyplot: Jupyter's inline backend loads matplotlib_inline,
# which breaks on newer matplotlib (RcParams has no _get). Agg avoids that.
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display


def show_figure(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight")
    buf.seek(0)
    display(Image(data=buf.getvalue()))
    plt.close(fig)


# DLL / GenICam env (must be before hsi_* imports; Jupyter does not source /etc/profile.d)
ROOT_FOLDER = os.path.abspath(r"/opt/imec/hsi-mosaic/")
if HOSTNAME == 'sage-303204':
    EBUS_SDK_ROOT = Path("/opt/pleora/ebus_sdk/Ubuntu-x86_64")
    if EBUS_SDK_ROOT.is_dir():
        genicam_root = EBUS_SDK_ROOT / "lib/genicam"
        os.environ.setdefault("PUREGEV_ROOT", str(EBUS_SDK_ROOT))
        os.environ.setdefault("GENICAM_ROOT", str(genicam_root))
        os.environ.setdefault("GENICAM_ROOT_V3_1", str(genicam_root))
        os.environ.setdefault(
            "GENICAM_LOG_CONFIG",
            str(genicam_root / "log/config/DefaultLogging.properties"),
        )
        os.environ.setdefault("GENICAM_LOG_CONFIG_V3_1", os.environ["GENICAM_LOG_CONFIG"])
        genicam_cache = Path.home() / ".config/Pleora/genicam_cache_v3_1"
        genicam_cache.mkdir(parents=True, exist_ok=True)
        os.environ.setdefault("GENICAM_CACHE", str(genicam_cache))
        os.environ.setdefault("GENICAM_CACHE_V3_1", str(genicam_cache))
        ld_paths = [
            str(EBUS_SDK_ROOT / "lib"),
            str(genicam_root / "bin/Linux64_x64"),
            str(Path(ROOT_FOLDER) / "bin"),
        ]
        existing_ld = os.environ.get("LD_LIBRARY_PATH", "")
        os.environ["LD_LIBRARY_PATH"] = os.pathsep.join(
            [p for p in ld_paths if p not in existing_ld.split(os.pathsep)]
            + ([existing_ld] if existing_ld else [])
        )

try:
    os.add_dll_directory(str(Path(ROOT_FOLDER) / "bin"))
except Exception:
    pass

try:
    CRNT_FOLDER = os.path.dirname(os.path.abspath(__file__))
except NameError:
    if HOSTNAME == 'hyper-desktop':
        CRNT_FOLDER = str(Path("/home/hyper/hyperspectral_camera/src").resolve())
    else:
        CRNT_FOLDER = str(Path("/home/campus.ncl.ac.uk/c4071391/Projects/hyperspectral_camera/src").resolve())

print(f'CRNT_FOLDER: {CRNT_FOLDER}')
# imec API modules use flat imports (e.g. "from hsi_common_types import *")
sys.path.append(ROOT_FOLDER + "/python_apis")
sys.path.append(ROOT_FOLDER + "/python_apis/hsi_common")
sys.path.append(ROOT_FOLDER + "/python_apis/hsi_mosaic")
sys.path.append(ROOT_FOLDER + "/python_apis/hsi_camera")
sys.path.append(ROOT_FOLDER + "/bin")

import hsi_mosaic as HSI_MOSAIC
import hsi_common as HSI_COMMON
import hsi_camera as HSI_CAMERA

logging.basicConfig(level=logging.INFO, format="%(message)s")

HSI_COMMON.InitializeLogger(
    str(Path(CRNT_FOLDER) / "logs"),
    HSI_COMMON.LoggerVerbosity.LV_VERBOSE,
)

print("COMMON API:", HSI_COMMON.GetAPIVersion())
print("CAMERA API:", HSI_CAMERA.GetAPIVersion())
print("MOSAIC API:", HSI_MOSAIC.GetAPIVersion())

vis_camera_model_string = 'MQ022HG-IM-SM4X4-VIS3'
nir_camera_model_string = 'MQ022HG-IM-SM5X5-NIR2'

# __file__ is undefined in Jupyter notebooks; fall back to cwd.
if "__file__" in globals():
    crnt_folder = os.path.abspath(os.path.dirname(__file__))
else:
    crnt_folder = os.getcwd()



In [ ]:
class ContextPaths:
    def __init__(self, data_dir):
        self.context_dir = data_dir + 'context/'
        self.calibration_file = self.try_collect_paths('calibration_file', first_result_only=True, prefix='')
        self.white_non_uniformity_file = self.try_collect_paths('non_uniformity', first_result_only=True, prefix='white_reference') 
        self.white_reference_file =  self.try_collect_paths('white_reference', first_result_only=True, prefix='white_reference')
        self.dark_reference_files =  self.try_collect_paths('dark_references', first_result_only=False, prefix='dark_reference')
        self.dark_non_uniformity_file =  self.dark_reference_files[0]
        self.output_dir = str(Path(data_dir).parent / 'processed')
        self.optical_setup = self.try_collect_paths('optical_setup', first_result_only=True, prefix='')
        print(f'Calibration file: {self.calibration_file}')
        print(f'White non-uniformity files: {self.white_non_uniformity_file}')
        print(f'White reference files: {self.white_reference_file}')
        print(f'Dark non-uniformity file: {self.dark_non_uniformity_file}')
        print(f'Dark reference files: {self.dark_reference_files}')
        print(f'Optical setup file: {self.optical_setup}')
        print(f'Output directory: {self.output_dir}')
        # assert os.path.exists(self.white_non_uniformity_file), f'White non-uniformity file does not exist: {self.white_non_uniformity_file}'
        # assert os.path.exists(self.white_reference_file), f'White reference file does not exist: {self.white_reference_file}'
        # assert os.path.exists(self.dark_non_uniformity_file), f'Dark non-uniformity file does not exist: {self.dark_non_uniformity_file}'
        # assert os.path.exists(Path(self.dark_reference_files[0])), f'Dark reference files do not exist: {self.dark_reference_file}'
        # assert os.path.exists(Path(self.calibration_file)), f'Calibration file does not exist: {self.calibration_file}'
    
    def try_collect_paths(self, folder_string, first_result_only=True, prefix=''):
        # first find all files in the folder
        if prefix == '':
            try:
                paths = [os.path.join(self.context_dir, folder_string, name) for name in os.listdir(self.context_dir + f'{folder_string}/')]
            except:
                return None
        else:
            try:
                paths = [os.path.join(self.context_dir, folder_string, name) for name in os.listdir(self.context_dir + f'{folder_string}') if prefix in name and '.raw.xml' in name]
            except:
                return None
        # then filter to single or whole list of files
        if paths == []:
            return None
        else:
            if first_result_only:
                return paths[0]
            else:
                return paths



# Calibration active areas (must match camera driver ROI):
#   VIS: 2048 x 1088
#   NIR: 2045 x 1085

# 18/06/2026
# run_data_directory = '/mnt/data/timelapses/20260618_123228_outdoor_white_ref_3'
# run_data_directory = '/mnt/data/hyper_payload_data/20260618_121242_outdoor_white_ref_1'
# run_data_directory = '/mnt/data/hyper_payload_data/20260618_122455_outdoor_white_ref_2'
# run_data_directory = '/mnt/data/hyper_payload_data/20260618_123228_outdoor_white_ref_3'
# run_data_directory = '/mnt/data/hyper_payload_data/20260618_124033_outdoor_white_ref_4'
# run_data_directory = '/mnt/data/hyper_payload_data/20260618_124611_outdoor_20hz'

# 24/06/2026 - on jetson
# run_data_directory = '/mnt/data/timelapses/20260625_163449_outdoor_strip_1'
# run_data_directory = '/mnt/data/timelapses/20260625_164919_outdoor_strip_2'

# 26/06/2026 - on desktop
run_data_directory = '/mnt/data/hyper_payload_data/20260625_164919_outdoor_strip_2/'

vis_data_directory = run_data_directory + '/vis/raw/'
nir_data_directory = run_data_directory + '/nir/raw/'

print(f'vis context files:')
vis_context_paths = ContextPaths(vis_data_directory)
print(f'nir context files:')
nir_context_paths = ContextPaths(nir_data_directory)


def _natural_acq_sort_key(path):
    parts = re.split(r'(\d+)', os.path.basename(path))
    return [int(part) if part.isdigit() else part for part in parts]


vis_camera_acq_files = sorted(
    (
        os.path.join(vis_data_directory, file)
        for file in os.listdir(vis_data_directory)
        if file.endswith(".raw.xml")
    ),
    key=_natural_acq_sort_key,
)


nir_camera_acq_files = sorted(
    (
        os.path.join(nir_data_directory, file)
        for file in os.listdir(nir_data_directory)
        if file.endswith(".raw.xml")
    ),
    key=_natural_acq_sort_key,
)

verbose = False

print(f'vis camera acq files:')
if verbose:
    for file in vis_camera_acq_files:
        print(file)

print(f'nir camera acq files:')
if verbose:
    for file in nir_camera_acq_files:
        print(file)

context = nir_context_paths
acquisition_files = nir_camera_acq_files


In [ ]:
# Fast multi frame implementation (VIS-ready; NIR needs ROI-cropped captures from updated driver)
class FastMultiFramePipeline():

    def __init__(self, context_paths, white_reference_roi, scene_acquisition_file, spatial_median_filter_enable=False, spatial_median_filter_kernel_size=3, white_balance_enable=True, deallocate=True, full_coverage=False):
        logger = logging.getLogger()
        logger.handlers.clear()
        logging.info("\n=========> Initialising pipeline and assigning white reference ****\n")

        self.context_paths = context_paths
        self.output_dir = context_paths.output_dir
        os.makedirs(self.output_dir, exist_ok=True)
        self.deallocate = deallocate
        self.scene_acquisition_file = scene_acquisition_file
        self.reference_frames = []
        self.context = None
        self.pipeline = None

        self.load_context(context_paths)
        self.pipeline = HSI_MOSAIC.Create(self.context)

        # get and adjust config if necessary
        pipeline_config = HSI_MOSAIC.GetConfigurationParameters(self.pipeline)
        pipeline_config['spatial_median_filter_enable']=spatial_median_filter_enable
        pipeline_config['spatial_median_filter_kernel_size']=spatial_median_filter_kernel_size
        pipeline_config['white_balance_enable']=white_balance_enable
        logging.info(f'Pipeline configuration parameters = {pipeline_config}')
        HSI_MOSAIC.SetConfigurationParameters(self.pipeline, pipeline_config)

        logging.info('Initialize Pipeline (irradiance stage)')
        HSI_MOSAIC.Initialize(self.pipeline)
        
        if full_coverage:
            white_reference_file = context_paths.white_non_uniformity_file
        else:
            white_reference_file = context_paths.white_reference_file

        logging.info(f'Loading white frame from file {white_reference_file}')
        white_frame = HSI_COMMON.LoadFrame(white_reference_file)
        white_cube = self.process_frame(white_frame)
        HSI_COMMON.DeallocateFrame(white_frame)

        if full_coverage:
            white_reference_roi = HSI_COMMON.RegionOfInterest(x=0, y=0, width=white_cube.width, height=white_cube.height)
            self.reference_spectrum = HSI_MOSAIC.ExtractSpectrumFromCube(white_cube, white_reference_roi)
            logging.info(f'Extracted reference spectrum from full coverage white cube ROI')
        else:
            self.reference_spectrum = HSI_MOSAIC.ExtractSpectrumFromCube(white_cube, white_reference_roi)
            logging.info(f'Extracted reference spectrum from {white_reference_roi} white cube ROI')
        HSI_COMMON.DeallocateCube(white_cube)

        HSI_MOSAIC.Stop(self.pipeline)
        HSI_MOSAIC.SetReferenceSpectrum(self.pipeline, self.reference_spectrum, 0.95)
        HSI_COMMON.DeallocateSpectrum(self.reference_spectrum)

        logging.info('Initialize Pipeline (normalization stage)')
        HSI_MOSAIC.Initialize(self.pipeline)
        self.outputdataformat = HSI_MOSAIC.GetOutputDataFormat(self.pipeline)
        logging.info(f'Output data format = {self.outputdataformat}')
        HSI_MOSAIC.Start(self.pipeline)

    def process_frame(self, input_frame):
        ''' 
        Allocate space for an output cube, based on the actual output data format.
        Process the given frame by pushing it through the pipeline
        and return the output cube
        Precondition: Pipeline state is READY 
        Note: the returned data cube must be deallocated by the caller
        '''

        # After initialization, the output data format can be queried
        outputdataformat = HSI_MOSAIC.GetOutputDataFormat(self.pipeline)
        logging.info(f'Got output data format = {outputdataformat}')

        # Now that the output data format is known, an empty output cube
        # with that format can be allocated.  This needs to be done
        # each time after a format (potentially) changed
        output_data_cube = HSI_COMMON.AllocateCube(outputdataformat)

        # Start the pipeline
        HSI_MOSAIC.Start(self.pipeline)

        # Process the frame
        HSI_MOSAIC.PushFrame(self.pipeline, input_frame)

        # Get the output cube from the pipeline
        HSI_MOSAIC.GetCube(self.pipeline, output_data_cube, timeout_ms=5000)
        logging.info(f'Produced output cube with format = {output_data_cube.format}')
        logging.info(f'Output cube info = {output_data_cube.info}')

        # The data of the output cube can be accessed by numpy as follows
        cube_numpy = HSI_COMMON.CubeAsArray(output_data_cube)
        logging.info(f'Cube shape = {cube_numpy.shape}, min = {cube_numpy.min()}, mean = {cube_numpy.mean()}, max = {cube_numpy.max()}')
        # Note that any change within the data in the numpy array will also be reflected in the output_data_cube 
        # class data, because the data memory is shared between the class and the numpy array

        # Put the pipeline back in READY state
        HSI_MOSAIC.Pause(self.pipeline)

        return output_data_cube

    def load_context(self, context_paths):
        '''Load context from disk; fall back to manual assembly for older captures.'''
        logging.info(f"Loading Context from {Path(context_paths.context_dir).absolute()}")
        assert Path(context_paths.context_dir).exists()

        scene_frame = HSI_COMMON.LoadFrame(self.scene_acquisition_file)
        logging.info(f'Scene frame shape = {HSI_COMMON.FrameAsArray(scene_frame).shape}')

        try:
            self.context = HSI_MOSAIC.LoadContext(context_paths.context_dir)
            logging.info('Context loaded via LoadContext')
            HSI_COMMON.DeallocateFrame(scene_frame)
            logging.info(f"Context status = {HSI_MOSAIC.ContextGetStatus(self.context)}")
            return
        except AssertionError as exc:
            logging.warning(f'LoadContext failed ({exc}); building context manually')

        self.context = HSI_MOSAIC.AllocateContext(scene_frame)
        HSI_MOSAIC.ContextSetCalibrationFile(self.context, context_paths.calibration_file)

        for dark_reference_file in context_paths.dark_reference_files:
            dark_reference_frame = HSI_COMMON.LoadFrame(dark_reference_file)
            self.reference_frames.append(dark_reference_frame)
        HSI_MOSAIC.ContextSetDarkFieldReferences(self.context, self.reference_frames)

        dark_non_uniformity_frame = HSI_COMMON.LoadFrame(context_paths.dark_non_uniformity_file)
        white_non_uniformity_frame = HSI_COMMON.LoadFrame(context_paths.white_non_uniformity_file)
        HSI_MOSAIC.ContextSetNonUniformity(
            self.context, dark_non_uniformity_frame, white_non_uniformity_frame)
        self.reference_frames.extend([dark_non_uniformity_frame, white_non_uniformity_frame])

        optical_setup = HSI_MOSAIC.LoadOpticalSetup(self.context_paths.optical_setup)
        HSI_MOSAIC.ContextSetOpticalSetup(self.context, optical_setup)
        HSI_COMMON.DeallocateFrame(scene_frame)

        status = HSI_MOSAIC.ContextGetStatus(self.context)
        logging.info(f"Context status = {status}")

    def push_frame(self, input_frame_file):
        '''Push one scene frame into the running normalization pipeline.'''
        input_frame = HSI_COMMON.LoadFrame(input_frame_file)
        HSI_MOSAIC.PushFrame(self.pipeline, input_frame)
        HSI_COMMON.DeallocateFrame(input_frame)

    def pull_and_save_frame(self, output_name):
        output_data_cube = HSI_COMMON.AllocateCube(self.outputdataformat)
        HSI_MOSAIC.GetCube(self.pipeline, output_data_cube, timeout_ms=30000)
        logging.info(f'Produced output cube with format = {output_data_cube.format}')

        cube_numpy = HSI_COMMON.CubeAsArray(output_data_cube)
        logging.info(
            f'Cube shape = {cube_numpy.shape}, min = {cube_numpy.min()}, '
            f'mean = {cube_numpy.mean()}, max = {cube_numpy.max()}')

        HSI_COMMON.SaveCube(
            output_data_cube, self.output_dir, output_name, HSI_COMMON.FileFormat.FF_ENVI)
        HSI_COMMON.DeallocateCube(output_data_cube)

    def close(self):
        if self.pipeline is not None:
            HSI_MOSAIC.Stop(self.pipeline)
            HSI_MOSAIC.Destroy(self.pipeline)
            self.pipeline = None
        for reference_frame in self.reference_frames:
            HSI_COMMON.DeallocateFrame(reference_frame)
        self.reference_frames = []
        if self.context is not None:
            HSI_MOSAIC.DeallocateContext(self.context)
            self.context = None

    def __del__(self):
        try:
            self.close()
        except Exception:
            pass



In [ ]:
# View example files

from matplotlib.patches import Rectangle


def _scaled_white_reference_roi(raw_xml_path):
    path = str(raw_xml_path).lower()

    # vis resolution: 512 x 272
    # mid point = 256 x 136
    vis_mid_point = {"x": 256, "y": 136}

    # nir resolution: 407 x 215
    # mid point = 204 x 108
    nir_mid_point = {"x": 204, "y": 108}

    ROI_WIDTH = 50
    ROI_HEIGHT = 50

    if "nir" in path:
        roi_scale = 5
        BASE_WHITE_REFERENCE_ROI = {"x": nir_mid_point["x"] * roi_scale - (ROI_WIDTH * roi_scale)/ 2 , "y": nir_mid_point["y"] * roi_scale - (ROI_HEIGHT * roi_scale)/ 2, "width": ROI_WIDTH * roi_scale, "height": ROI_HEIGHT * roi_scale}
    elif "vis" in path:
        roi_scale = 4
        BASE_WHITE_REFERENCE_ROI = {"x": vis_mid_point["x"] * roi_scale - (ROI_WIDTH * roi_scale)/ 2 , "y": vis_mid_point["y"] * roi_scale - (ROI_HEIGHT * roi_scale)/ 2, "width": ROI_WIDTH * roi_scale, "height": ROI_HEIGHT * roi_scale}    
    return BASE_WHITE_REFERENCE_ROI, roi_scale


def view_raw_file(raw_xml_path, title=None):
    """Load an HSI .raw.xml file and display a quick sanity-check preview."""
    print(f"Loading: {raw_xml_path}")
    frame = HSI_COMMON.LoadFrame(raw_xml_path)
    try:
        print(f"  format: {frame.format}")
        print(f"  info:   {frame.info}")
        arr = HSI_COMMON.FrameAsArray(frame)
        print(f"  shape:  {arr.shape}")
        print(f"  dtype:  {arr.dtype}")
        print(f"  min/mean/max: {arr.min():.2f} / {arr.mean():.2f} / {arr.max():.2f}")

        roi, roi_scale = _scaled_white_reference_roi(raw_xml_path)
        print(
            "  white ref ROI scaled up:"
            f" x={roi['x']}, y={roi['y']},"
            f" width={roi['width']}, height={roi['height']}"
        )
        print(
            "  white ref ROI actual:"
            f" x={roi['x']/roi_scale}, y={roi['y']/roi_scale},"
            f" width={roi['width']/roi_scale}, height={roi['height']/roi_scale}"
        )

        img = np.squeeze(arr)
        if img.ndim > 2:
            img = np.mean(img, axis=-1)

        fig, axes = plt.subplots(
            1, 2, figsize=(12, 5), gridspec_kw={"width_ratios": [3, 1]}
        )
        im = axes[0].imshow(img, cmap="gray", vmin=0, vmax=255)
        axes[0].add_patch(
            Rectangle(
                (roi["x"], roi["y"]),
                roi["width"],
                roi["height"],
                linewidth=2,
                edgecolor="lime",
                facecolor="none",
            )
        )
        axes[0].set_title(title or os.path.basename(raw_xml_path))
        # axes[0].axis("off")
        fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

        axes[1].hist(img.ravel(), bins=256, color="tab:blue", alpha=0.8)
        axes[1].set_title("Intensity histogram")
        axes[1].set_xlabel("Pixel intensity")
        axes[1].set_ylabel("Count")
        axes[1].set_xlim(0, 1023)
        axes[1].grid(alpha=0.2)

        plt.tight_layout()
        show_figure(fig)
    finally:
        HSI_COMMON.DeallocateFrame(frame)


def _format_scale_ratio(ratio):
    """Compact ratio string for scaled output filenames."""
    return f"{ratio:.6g}"


def scale_non_uniformity_by_integration_time(
    raw_xml_path, desired_integration_time_ms, output_dir=None
):
    """Scale a non-uniformity .raw.xml to a target integration time.

    Reads the actual integration_time_ms from frame info, scales pixel
    intensities by desired/actual, updates metadata, and saves a new
    .raw.xml alongside the source file with suffix ``scaled_<ratio>``.
    """
    raw_xml_path = str(raw_xml_path)
    desired_integration_time_ms = float(desired_integration_time_ms)

    print(f"Loading: {raw_xml_path}")
    frame = HSI_COMMON.LoadFrame(raw_xml_path)
    try:
        actual_integration_time_ms = float(frame.info["integration_time_ms"])
        if actual_integration_time_ms <= 0:
            raise ValueError(
                f"Invalid actual integration_time_ms: {actual_integration_time_ms}"
            )
        if desired_integration_time_ms <= 0:
            raise ValueError(
                f"Invalid desired integration_time_ms: {desired_integration_time_ms}"
            )

        ratio = desired_integration_time_ms / actual_integration_time_ms
        ratio_str = _format_scale_ratio(ratio)

        arr = HSI_COMMON.FrameAsArray(frame)
        print(
            f"  actual integration_time_ms: {actual_integration_time_ms}"
            f", desired: {desired_integration_time_ms}"
            f", scale ratio: {ratio_str}"
        )
        print(
            f"  intensity before min/mean/max:"
            f" {arr.min():.2f} / {arr.mean():.2f} / {arr.max():.2f}"
        )

        arr *= ratio
        value_max = float(frame.info["value_max"])
        if value_max > 0:
            np.clip(arr, 0.0, value_max, out=arr)

        frame.info.integration_time_ms = desired_integration_time_ms

        print(
            f"  intensity after  min/mean/max:"
            f" {arr.min():.2f} / {arr.mean():.2f} / {arr.max():.2f}"
        )

        source_path = Path(raw_xml_path)
        stem = source_path.name
        if stem.endswith(".raw.xml"):
            stem = stem[: -len(".raw.xml")]
        elif stem.endswith(".xml"):
            stem = stem[: -len(".xml")]

        out_dir = Path(output_dir) if output_dir is not None else source_path.parent
        out_dir.mkdir(parents=True, exist_ok=True)
        out_stem = f"{stem}_scaled_{ratio_str}"

        HSI_COMMON.SaveFrame(
            frame, str(out_dir), out_stem, HSI_COMMON.FileFormat.FF_RAW
        )
        out_path = out_dir / f"{out_stem}.raw.xml"
        print(f"  saved: {out_path}")
        return str(out_path)
    finally:
        HSI_COMMON.DeallocateFrame(frame)


raw_file_to_view = context.white_non_uniformity_file
# raw_file_to_view = '/mnt/data/timelapses/20260625_163449_outdoor_strip_1/vis/raw/context/non_uniformity/white_reference_7.raw.xml'
view_raw_file(raw_file_to_view)

# # Example: scale white non-uniformity from 10 ms capture to 4 ms equivalent
# scaled_non_uniformity_file = scale_non_uniformity_by_integration_time(
#     context.white_non_uniformity_file,
#     desired_integration_time_ms=7.0,
# )

# view_raw_file(scaled_non_uniformity_file)






In [ ]:
%time
%timeit

# VIS processing (NIR requires ROI-cropped 2045x1085 captures from the updated camera driver)

# ROI is relative to the pipeline output cube (default 510x270 for VIS), not the raw frame
white_reference_roi = HSI_COMMON.RegionOfInterest(x=231, y=111, width=50, height=50)


mfp = FastMultiFramePipeline( 
    context, 
    white_reference_roi, 
    acquisition_files[0], 
    spatial_median_filter_enable=False, 
    spatial_median_filter_kernel_size=0, 
    white_balance_enable=True, 
    deallocate=True 
    )

file_limit = -1

idx = 0
for file in tqdm.tqdm(acquisition_files):
    # if idx % 20 == 0:
    mfp.push_frame(file) # note can only seem to hold 10 frames in memory before needing to have them pulled. 
    stem = Path(file).name.replace('.raw.xml', '')
    # print(f"Pushing frame: {stem}")
    mfp.pull_and_save_frame(f'normalized_{stem}')
    idx += 1
    if idx >= file_limit and file_limit > 0:
        break

mfp.close()
del mfp

# about 2.11 It/s, no GPU use
#  


In [ ]:
import spectral as sp

processed_paths = [os.path.join(context.output_dir, file) 
    for file in os.listdir(context.output_dir) 
    if file.endswith(".hdr")]

for idx, path in enumerate(processed_paths):
    if idx % 20 == 0:
        print(f"Opening: {path}")

        img = sp.open_image(path)
        cube = img.load()
        wavelength = np.asarray(img.bands.centers, dtype=float)
        print(f"Cube shape (lines, samples, bands): {cube.shape}")
        print(f"Wavelengths (nm): {wavelength}")

        band_index = cube.shape[-1] // 2
        band = cube[:, :, band_index]

        # Sample reflectance spectra from a regular grid across the scene
        curves = 10
        rows = 2
        height, width = cube.shape[:2]
        row_step = max(1, height // (rows + 1))
        col_step = max(1, width // ((curves // rows) + 1))
        sample_curves_cube = cube[row_step::row_step, col_step::col_step, :]
        sample_curves = sample_curves_cube.reshape(-1, cube.shape[-1])

        # False-colour RGB using VIS band indices (R, G, B)
        rgb_bands = (14, 10, 5) if cube.shape[-1] > 14 else (cube.shape[-1] - 1, cube.shape[-1] // 2, 0)
        rgb = np.clip(cube[:, :, rgb_bands], 0, 1)

        fig, ax = plt.subplots(2, 2, figsize=(12, 10))
        ax[0, 0].imshow(band, cmap="gray", vmin=0, vmax=1)
        ax[0, 0].set_title(f"Band {band_index} ({wavelength[band_index]:.1f} nm)")
        ax[0, 1].imshow(rgb)
        ax[0, 1].set_title(f"False colour (bands {rgb_bands})")
        for curve in sample_curves:
            ax[1, 0].plot(wavelength, curve)
        ax[1, 0].set_title(f"{len(sample_curves)} sample spectra")
        ax[1, 0].set_xlabel("Wavelength (nm)")
        ax[1, 0].set_ylabel("Reflectance")
        ax[1, 1].plot(wavelength, cube.mean(axis=(0, 1)))
        ax[1, 1].set_title("Mean spectrum")
        ax[1, 1].set_xlabel("Wavelength (nm)")
        ax[1, 1].set_ylabel("Reflectance")

        plt.tight_layout()
        show_figure(fig)

    if idx >= 10000:
        break


In [ ]:
import os
print(os.environ.get("LD_LIBRARY_PATH", "<empty>"))

# if ROS environment does not show here run 'killall node' on the remote server and reconnect

In [ ]:
# unpack rosbag
# Jupyter / Cursor Remote SSH do not source setup.bash — bootstrap ROS env here.

import csv
import json
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
from rosbag2_py import SequentialReader, StorageOptions, ConverterOptions
from rclpy.serialization import deserialize_message
from rosidl_runtime_py.utilities import get_message


def _topic_to_dir(topic: str) -> str:
    return topic.strip("/").replace("/", "__") or "root"


def _image_msg_to_array(msg):
    height, width = msg.height, msg.width
    if msg.encoding == "rgb8":
        arr = np.frombuffer(msg.data, dtype=np.uint8).reshape((height, width, 3))
        return cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)
    if msg.encoding == "bgr8":
        return np.frombuffer(msg.data, dtype=np.uint8).reshape((height, width, 3))
    if msg.encoding == "mono8":
        return np.frombuffer(msg.data, dtype=np.uint8).reshape((height, width))
    if msg.encoding in ("mono16", "16UC1"):
        return np.frombuffer(msg.data, dtype=np.uint16).reshape((height, width))
    raise ValueError(f"Unsupported image encoding: {msg.encoding}")


def _parse_key_value_lines(data: str) -> dict[str, str]:
    parsed = {}
    for line in data.strip().splitlines():
        line = line.strip()
        if not line or "=" not in line:
            continue
        key, _, value = line.partition("=")
        parsed[key.strip()] = value.strip()
    return parsed


def _is_key_value_text(data: str) -> bool:
    lines = [line.strip() for line in data.strip().splitlines() if line.strip()]
    return len(lines) > 1 and all("=" in line for line in lines)


_SPECTROMETER_CHANNELS = [
    "415nm", "445nm", "480nm", "515nm", "555nm",
    "590nm", "630nm", "680nm", "nir", "clear",
]


def _csv_row_from_string(msg, timestamp_ns: int) -> tuple[list[str], dict]:
    if _is_key_value_text(msg.data):
        fields = _parse_key_value_lines(msg.data)
        return ["timestamp_ns"] + list(fields.keys()), {"timestamp_ns": timestamp_ns, **fields}
    return ["timestamp_ns", "data"], {"timestamp_ns": timestamp_ns, "data": msg.data}


def _csv_row_from_nav_sat_fix(msg, timestamp_ns: int) -> tuple[list[str], dict]:
    fieldnames = [
        "timestamp_ns",
        "header_stamp_sec",
        "header_stamp_nsec",
        "frame_id",
        "latitude",
        "longitude",
        "altitude",
        "status",
        "service",
        "position_covariance_type",
    ]
    row = {
        "timestamp_ns": timestamp_ns,
        "header_stamp_sec": msg.header.stamp.sec,
        "header_stamp_nsec": msg.header.stamp.nanosec,
        "frame_id": msg.header.frame_id,
        "latitude": msg.latitude,
        "longitude": msg.longitude,
        "altitude": msg.altitude,
        "status": msg.status.status,
        "service": msg.status.service,
        "position_covariance_type": msg.position_covariance_type,
    }
    return fieldnames, row


def _csv_row_from_float32_multiarray(msg, timestamp_ns: int, topic: str) -> tuple[list[str], dict]:
    if topic == "/sensors/spectrometer" and len(msg.data) == len(_SPECTROMETER_CHANNELS):
        names = _SPECTROMETER_CHANNELS
    else:
        names = [f"ch{i}" for i in range(len(msg.data))]
    row = {"timestamp_ns": timestamp_ns}
    for name, value in zip(names, msg.data):
        row[name] = value
    return ["timestamp_ns"] + names, row


def unpack_rosbag(
    bag_dir: str | Path,
    output_dir: str | Path | None = None,
    skip_topics: set[str] | None = None,
    only_topics: str | set[str] | None = None,
) -> Path:
    """Extract rosbag topics into per-topic folders of PNGs and CSVs.

    If only_topics is set, unpack only those topic(s) and skip all others.
    """
    bag_dir = Path(bag_dir)
    if output_dir is None:
        output_dir = bag_dir.parent / "rosbag_unpacked"
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    skip_topics = skip_topics or {
        "/rosout",
        "/parameter_events",
        "/events/write_split",
    }
    if only_topics is not None:
        only_topics = {only_topics} if isinstance(only_topics, str) else set(only_topics)

    reader = SequentialReader()
    reader.open(
        StorageOptions(uri=str(bag_dir), storage_id="sqlite3"),
        ConverterOptions(
            input_serialization_format="cdr",
            output_serialization_format="cdr",
        ),
    )
    topic_types = {t.name: t.type for t in reader.get_all_topics_and_types()}

    csv_files = {}
    csv_writers = {}
    image_counters = defaultdict(int)
    summary = defaultdict(int)
    unhandled_types = defaultdict(int)

    def _get_csv_writer(topic, fieldnames):
        if topic not in csv_writers:
            topic_dir = output_dir / _topic_to_dir(topic)
            topic_dir.mkdir(parents=True, exist_ok=True)
            f = open(topic_dir / "messages.csv", "w", newline="", encoding="utf-8")
            csv_files[topic] = f
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            csv_writers[topic] = writer
        return csv_writers[topic]

    def _write_csv_row(topic, fieldnames, row):
        _get_csv_writer(topic, fieldnames).writerow(row)
        summary[topic] += 1

    while reader.has_next():
        topic, data, timestamp_ns = reader.read_next()
        if topic in skip_topics:
            continue
        if only_topics is not None and topic not in only_topics:
            continue

        msg_type = topic_types[topic]
        msg = deserialize_message(data, get_message(msg_type))

        if msg_type == "sensor_msgs/msg/Image":
            topic_out = output_dir / _topic_to_dir(topic)
            topic_out.mkdir(parents=True, exist_ok=True)
            arr = _image_msg_to_array(msg)
            idx = image_counters[topic]
            image_counters[topic] += 1
            out_path = topic_out / f"{timestamp_ns:019d}_{idx:06d}.png"
            cv2.imwrite(str(out_path), arr)
            _write_csv_row(
                topic,
                ["timestamp_ns", "frame_index", "filename", "width", "height", "encoding"],
                {
                    "timestamp_ns": timestamp_ns,
                    "frame_index": idx,
                    "filename": out_path.name,
                    "width": msg.width,
                    "height": msg.height,
                    "encoding": msg.encoding,
                },
            )

        elif msg_type == "std_msgs/msg/Float32":
            _write_csv_row(
                topic,
                ["timestamp_ns", "value"],
                {"timestamp_ns": timestamp_ns, "value": msg.data},
            )

        elif msg_type == "std_msgs/msg/String":
            fieldnames, row = _csv_row_from_string(msg, timestamp_ns)
            _write_csv_row(topic, fieldnames, row)

        elif msg_type == "std_msgs/msg/Float32MultiArray":
            fieldnames, row = _csv_row_from_float32_multiarray(msg, timestamp_ns, topic)
            _write_csv_row(topic, fieldnames, row)

        elif msg_type == "sensor_msgs/msg/NavSatFix":
            fieldnames, row = _csv_row_from_nav_sat_fix(msg, timestamp_ns)
            _write_csv_row(topic, fieldnames, row)

        else:
            unhandled_types[msg_type] += 1

    for f in csv_files.values():
        f.close()

    summary_path = output_dir / "unpack_summary.json"
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump({"bag_dir": str(bag_dir), "topics": dict(summary)}, f, indent=2)

    print(f"Unpacked to: {output_dir}")
    for topic, count in sorted(summary.items()):
        print(f"  {topic}: {count} messages")
    if unhandled_types:
        print("Skipped unhandled message types:")
        for msg_type, count in sorted(unhandled_types.items()):
            print(f"  {msg_type}: {count} messages")
    return output_dir


ROSBAG_DIR = f"{run_data_directory}/rosbag"
unpack_output_dir = unpack_rosbag(
    ROSBAG_DIR,
    output_dir=f"{run_data_directory}/rosbag_unpacked"
)


In [ ]:
# make rgb images into movie

def png_folder_to_mp4(png_dir, out_path=None, fps=20.0, fourcc="mp4v"):
    """Write sorted PNGs in a folder to an MP4 (BGR frames, as saved by unpack_rosbag)."""
    png_dir = Path(png_dir)
    frames = sorted(png_dir.glob("*.png"), key=lambda p: p.name)
    if not frames:
        raise FileNotFoundError(f"No PNGs in {png_dir}")

    if out_path is None:
        out_path = png_dir / "movie.mp4"
    out_path = Path(out_path)

    first = cv2.imread(str(frames[0]))
    if first is None:
        raise RuntimeError(f"Failed to read {frames[0]}")
    h, w = first.shape[:2]

    writer = cv2.VideoWriter(
        str(out_path),
        cv2.VideoWriter_fourcc(*fourcc),
        fps,
        (w, h),
    )
    if not writer.isOpened():
        raise RuntimeError(f"Failed to open VideoWriter for {out_path}")

    for frame_path in frames:
        img = cv2.imread(str(frame_path))
        if img is None:
            raise RuntimeError(f"Failed to read {frame_path}")
        writer.write(img)
    writer.release()

    print(f"Wrote {len(frames)} frames @ {fps} fps -> {out_path}")
    return out_path


left_stereo_dir = unpack_output_dir / "sensors__stereo__left__throttled"
left_stereo_movie = png_folder_to_mp4(left_stereo_dir, fps=20.0)